# figure5.ipynb

Manuscript analysis and plotting notebook.

Run the configuration cell first. Input data and fonts are not bundled.
See the repository README and data requirements before executing.


In [ ]:
from pathlib import Path
import sys as _sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'plot_config.py').is_file()
     and (p / 'config.example.json').is_file()),
    None,
)
if _repo_root is None:
    raise RuntimeError('Run this notebook from within the plotting repository.')
if str(_repo_root) not in _sys.path:
    _sys.path.insert(0, str(_repo_root))
from plot_config import configured_path


In [ ]:
import numpy as np  
import matplotlib.pyplot as plt  
import matplotlib.patches as patches 
import xarray as xr
import pandas as pd
import cartopy.crs as ccrs  
import cartopy.feature as cfeature
import cmaps
from cartopy.util import add_cyclic_point
import matplotlib.ticker as ticker
import matplotlib.dates as mdates

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from pathlib import Path
from matplotlib import font_manager
font_dir = Path(configured_path('font_dir'))
font_files = [font_dir / 'Times New Roman.ttf', font_dir / 'Times New Roman Bold.ttf']
for font_file in font_files:
    font_manager.fontManager.addfont(str(font_file))
assert Path(font_manager.findfont('Times New Roman', fallback_to_default=False)).resolve() in {p.resolve() for p in font_files}


In [ ]:
fdir1 = configured_path('reccap_mask_file')
mask = xr.open_mfdataset(fdir1)
mask_ocean = mask.open_ocean

In [ ]:
fdir1 = configured_path('paired_mld_file')
ds = xr.open_dataset(fdir1)
# Retain the original January 1982 start, with one timestamp per sample.
time_new = pd.date_range(start='1982-01-01', periods=ds.sizes['time'], freq='MS') + pd.Timedelta(days=14)
ds = ds.assign_coords(time=time_new)


In [ ]:
fdir2 = configured_path('argo_climatology_file')
ds2 = xr.open_mfdataset(fdir2)
# 1. 对lon数值做 +360 转换（负数经度加360）
ds2["lon"] = xr.where(ds2.lon < 0, ds2.lon + 360, ds2.lon)
# 2. 把数据集按照新的lon从小到大重新排序，实现0.5~359.5
ds2 = ds2.sortby("lon")
ds2 = ds2.set_coords(["lat", "lon"])
ds2

In [ ]:
pred_clim = ds.MLD_pred.where(mask_ocean).groupby('time.month').mean()
#mom6_clim = ds2.MLD_003.where(mask_ocean).groupby('time.month').mean()

In [ ]:
fdir3 = configured_path('de_boyer_climatology_file')
ds3 = xr.open_mfdataset(fdir3)
# 新建month坐标，覆盖原有time维度
month_vals = np.arange(1, 13)  # [1,2,...,12]
ds3 = ds3.assign_coords(month=("time", month_vals))
# 将维度名称从time重命名为month
ds3 = ds3.swap_dims({"time": "month"})
# 删除旧time坐标（可选，清理结构）
ds3 = ds3.drop_vars("time")
ds3

In [ ]:
cmap1 = cmaps.precip_11lev[1:]

In [ ]:
cmap2 = cmap1.interp(12)

In [ ]:
def shift_lon_30_390(da, lon_name="lon"):
    new_lon = ((da[lon_name] - 30) % 360) + 30
    return da.assign_coords({lon_name: new_lon}).sortby(lon_name)

pred_clim = shift_lon_30_390(pred_clim)
ds2 = shift_lon_30_390(ds2)
ds3 = shift_lon_30_390(ds3)


In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as ticker
from matplotlib.colors import BoundaryNorm
import xarray as xr
from cartopy.mpl.gridliner import LongitudeFormatter, LatitudeFormatter

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Times New Roman'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['mathtext.bf'] = 'Times New Roman:bold'
plt.rcParams['axes.unicode_minus'] = False

# 非线性不等分色阶
nonlinear_levels = [
    10, 15, 25, 40, 60, 80, 100,
    125, 200, 300, 500, 700
]

# 四季配置
season_configs = [
    {"months_idx": [11, 0, 1], "title": "DJF"},
    {"months_idx": [2, 3, 4], "title": "MAM"},
    {"months_idx": [5, 6, 7], "title": "JJA"},
    {"months_idx": [8, 9, 10], "title": "SON"},
]

# 三套数据配置
data_sets = [
    {
        "data": pred_clim,
        "name": "SJTU-ViT",
        "var": None,
        "month_dim": "month",
        "need_xy": False,
    },
    {
        "data": ds2,
        "name": "Obs(Argo)",
        "var": "mld_dt_mean",
        "month_dim": "iMONTH",
        "need_xy": True,
    },
    {
        "data": ds3,
        "name": "Obs(de Boyer)",
        "var": "mld_dr003",
        "month_dim": "month",
        "need_xy": False,
    },
]

n_season = len(season_configs)
n_data = len(data_sets)

# 创建 3 行 4 列 Robinson 投影子图
fig, axes = plt.subplots(
    nrows=n_data,
    ncols=n_season,
    figsize=(26, 10),
    subplot_kw={
        "projection": ccrs.Robinson(central_longitude=210)
    },
)

# 保证 axes 始终为二维数组
axes = axes.reshape(n_data, n_season)

cmap = cmap2
norm = BoundaryNorm(
    nonlinear_levels,
    ncolors=cmap.N,
    extend="max",
)

plot_collection = None

# 行表示数据，列表示季节
for row_idx, ds_info in enumerate(data_sets):
    raw_ds = ds_info["data"]
    data_name = ds_info["name"]
    var_name = ds_info["var"]
    dim_name = ds_info["month_dim"]
    need_xy = ds_info["need_xy"]

    for col_idx, season in enumerate(season_configs):
        ax = axes[row_idx, col_idx]

        # 提取目标变量
        if var_name is None:
            data_array = raw_ds
        else:
            data_array = raw_ds[var_name]

        # 计算季节平均
        mld_season = (
            data_array
            .isel({dim_name: season["months_idx"]})
            .mean(dim=dim_name)
        )

        # Keep the sorted native longitude grid: adding a cyclic column here
        # causes incorrect Cartopy contour polygon filling across the map seam.
        ax.set_global()

        # 绘图参数
        plot_kwargs = {
            "levels": nonlinear_levels,
            "ax": ax,
            "transform": ccrs.PlateCarree(),
            "cmap": cmap,
            "norm": norm,
            "extend": "max",
            "add_colorbar": False,
        }

        # ds2 需要明确指定经纬度坐标
        if need_xy:
            plot_kwargs["x"] = "lon"
            plot_kwargs["y"] = "lat"

        plot_collection = mld_season.plot.contourf(**plot_kwargs)

        # 地图要素
        ax.coastlines(linewidth=0.8, zorder=3)
        ax.add_feature(
            cfeature.LAND,
            facecolor="dimgray",
            edgecolor="none",
            zorder=2,
        )

        # 经纬网格
        gl = ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            x_inline=False,
            y_inline=False,
            linewidth=0.7,
            color="gray",
            alpha=0.6,
            linestyle="-",
            zorder=1,
        )
        
        # 经线位置：地图中央为 150°W
        gl.xlocator = ticker.FixedLocator([
            30, 90, 150, -150, -90, -30
        ])
        
        # 不再显示 60°N 和 60°S
        gl.ylocator = ticker.FixedLocator([
            -45, 0, 45
        ])
        
        gl.yformatter = LatitudeFormatter(
            degree_symbol="°",
        )
        
        # 关闭 Cartopy 自动经度标签
        gl.top_labels = False
        gl.bottom_labels = False
        gl.right_labels = False
        
        # 仅最左侧一列显示纬度标签
        gl.left_labels = col_idx == 0
        
        gl.ylabel_style = {
            "size": 22,
            "color": "black",
        }
        
        # 最下面一行手动添加对称经度标签
        if row_idx == n_data - 1:
            longitude_labels = [
                (0.24, "30°E"),
                (0.42, "150°E"),
                (0.60, "90°W"),
                (0.76, "30°E"),
            ]
        
            for x_position, label in longitude_labels:
                ax.text(
                    x_position,
                    -0.025,
                    label,
                    transform=ax.transAxes,
                    ha="center",
                    va="top",
                    fontsize=22,
                    clip_on=False,
                )


        # 第一行添加季节标题
        if row_idx == 0:
            ax.set_title(
                season["title"],
                pad=10,
                fontsize=27,
            )
        else:
            ax.set_title("")

        # 第一列添加数据名称
        if col_idx == 0:
            ax.text(
                -0.13,
                0.5,
                data_name,
                transform=ax.transAxes,
                fontsize=27,
                va="center",
                ha="center",
                rotation=90,
            )

# 右侧统一色条
cax = fig.add_axes([0.95, 0.24, 0.009, 0.56])

cbar = fig.colorbar(
    plot_collection,
    cax=cax,
    ticks=nonlinear_levels,
    format=ticker.ScalarFormatter(),
    spacing="uniform",
    extend="max",
)

cbar.set_label(
    "Mixed Layer Depth (m)",
    labelpad=14,
    fontsize=22.5,
)

cbar.ax.tick_params(labelsize=22.5)

# 调整子图间距
fig.subplots_adjust(
    left=0.05,
    right=0.94,
    top=0.93,
    bottom=0.06,
    wspace=0.0,
    hspace=0.06,
)

output_path = Path(str(Path(configured_path('output_dir')) / 'figure5.png'))
output_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_path, dpi=600, bbox_inches='tight', pad_inches=0, facecolor='white')
print(f'Saved: {output_path}')

plt.show()
plt.close(fig)